In [5]:
%pip install scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 39.6 MB/s  0:00:00m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 24.5 MB/s  0:00:00m0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 30.4 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [scikit-learn] [scikit-learn]
Note: you may need to restart the kernel to use updated packages.


In [7]:
%pip install pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 38.4 MB/s  0:00:006m0:00:01
Note: you may need to restart the kernel to use updated packages.


In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("insurance.csv")  # load the data
y = df['charges']  # what we're predicting
X = pd.get_dummies(df.drop(columns='charges'), drop_first=True).astype(float)  # turn text into 0/1 columns

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)  

# just scaling the variables since we'll need to anyway with PCA
scaler = StandardScaler()  
X_train_s = scaler.fit_transform(X_train) 
X_test_s = scaler.transform(X_test)  

In [10]:
# Forward Selection

from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import LinearRegression

selector = SequentialFeatureSelector(LinearRegression(), n_features_to_select=4, direction='forward')
selector.fit(X_train_s, y_train)

forward_chosen = X.columns[selector.get_support()]  # which features got picked
print(forward_chosen)

Index(['age', 'bmi', 'children', 'smoker_yes'], dtype='str')


In [ ]:
# Backward 
backward = SequentialFeatureSelector(LinearRegression(), n_features_to_select=4, direction='backward') 
backward.fit(X_train_s, y_train)

chosen_backward = X.columns[backward.get_support()]  # which features survived
print(chosen_backward)

Index(['age', 'bmi', 'children', 'smoker_yes'], dtype='str')


In [12]:
# PCA 
from sklearn.decomposition import PCA

pca = PCA()  # no number specified yet — just see what's possible
pca.fit(X_train_s)

print(pca.explained_variance_ratio_)

[0.18645863 0.16641183 0.14048665 0.13335965 0.12060724 0.11274962
 0.09983156 0.04009483]


In [13]:
import numpy as np
print(np.cumsum(pca.explained_variance_ratio_))

[0.18645863 0.35287046 0.49335711 0.62671676 0.747324   0.86007362
 0.95990517 1.        ]


In [14]:
loadings = pd.DataFrame(
    pca.components_.T,  # flip it so features are rows, components are columns
    columns=[f'PC{i+1}' for i in range(pca.n_components_)],  # label columns PC1, PC2, ...
    index=X.columns  # label rows with your original feature names
)
print(loadings.round(2))

                   PC1   PC2   PC3   PC4   PC5   PC6   PC7   PC8
age               0.05  0.07  0.71  0.20 -0.20  0.38 -0.51  0.01
bmi               0.47  0.06  0.35  0.10 -0.25  0.10  0.74 -0.14
children         -0.06 -0.04  0.33  0.48  0.73 -0.33  0.11 -0.00
sex_male          0.00 -0.08 -0.18  0.66 -0.55 -0.46 -0.13  0.01
smoker_yes        0.10 -0.10 -0.44  0.51  0.17  0.70  0.04  0.01
region_northwest -0.49 -0.55  0.19  0.01 -0.15  0.13  0.30  0.54
region_southeast  0.70 -0.23 -0.07 -0.09  0.12 -0.13 -0.22  0.61
region_southwest -0.16  0.79 -0.03  0.10 -0.03  0.03  0.14  0.56


In [15]:
pca = PCA(n_components=7)  # pick a number from what you saw in step 4
X_train_pca = pca.fit_transform(X_train_s)  # blend the train features
X_test_pca = pca.transform(X_test_s)  # apply same blend to test

pcr_model = LinearRegression().fit(X_train_pca, y_train)  # regular regression on the blended features
print("PCR Test R²:", pcr_model.score(X_test_pca, y_test))

PCR Test R²: 0.7809931526960909


In [16]:
from sklearn.cross_decomposition import PLSRegression

pls = PLSRegression(n_components=7)  # same number, for a fair comparison
pls.fit(X_train_s, y_train)
print("PLSR Test R²:", pls.score(X_test_s, y_test))

PLSR Test R²: 0.7835930506681749


In [ ]:
#